# Landau damping study in an open, grounded plasma

We solve the **nonlinear, self-consistent electron Vlasov–Poisson system**
$$f_t+v f_z-E f_v=0,\qquad \phi_{zz}=n_e-1,\qquad E=-\phi_z,
\qquad n_e=\int f\,dv.$$
Ions are stationary with density 1. Units satisfy $v_{te}=\omega_{pe}=\lambda_D=1$.
The domain is $z\in[-40,40]$, $v\in[-6,6]$; both endpoint potentials are zero.

The left boundary supplies a Maxwellian only for $v>0$, the right only for
$v<0$. Outgoing particles escape freely. Velocity-boundary inflow is also
Maxwellian, selected locally by the sign of $-E(z,t)$. The truncated Maxwellian
is normalized with BSPF integration so the numerical equilibrium has density 1.
These are grounded reservoirs, not periodic or transparent field boundaries.

A small localized density perturbation excites several wavenumbers around
$k_0=0.5$. We assess phase mixing, field–particle energy transfer, boundary
loss and convergence. A falling electric field alone is not sufficient evidence
of Landau damping, and this wave packet has no single prescribed damping rate.

Install `python -m pip install -e './jax[test,notebook]'` and select that kernel.


In [ ]:
import jax
jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt
import bspf_jax as b

amplitude, carrier = 1e-3, 0.5
center, width = 3., 24.
times = jnp.linspace(0., 12., 121)

def solve(nz=129, nv=129, half_length=40., substeps=5):
    z = jnp.linspace(-half_length, half_length, nz)
    v = jnp.linspace(-6., 6., nv)
    zp = b.plan_1d(z, degree=7, n_basis=24, boundary_points=9)
    vp = b.plan_1d(v, degree=7, n_basis=24, boundary_points=9)
    model = b.plan_vlasov_poisson(zp, vp, temperature=1., quadrature_order=8)
    r = (z-center)/width
    window = jnp.where(jnp.abs(r) < 1, jnp.cos(jnp.pi*r/2)**8, 0.)
    shape = window*jnp.cos(carrier*(z-center))
    shape -= window*b.integrate(zp, shape)/b.integrate(zp, window)
    initial = model.background[None, :]*(1+amplitude*shape[:, None])
    f, phi, electric = b.integrate_vlasov_poisson(
        model, initial, times, substeps=substeps)
    return z, v, zp, vp, model, f, phi, electric

z, v, zp, vp, model, f, phi, electric = solve()
delta = f-model.background
field_energy = 0.5*b.integrate(zp, (electric**2).T)
print(f"Minimum distribution: {float(jnp.min(f)):.3e}")
print(f"Grounded potential residual: {float(jnp.max(jnp.abs(phi[:, jnp.array([0, -1])]))):.3e}")
print(f"Final / initial field energy: {float(field_energy[-1]/field_energy[0]):.6f}")
assert bool(jnp.all(jnp.isfinite(f))) and float(jnp.min(f)) > 0
assert float(jnp.max(jnp.abs(phi[:, jnp.array([0, -1])]))) < 1e-13


## Poisson from BSPF antiderivatives at every RK stage

For $s=n_e-1$, form $G(z)=\int_{z_L}^z s(y)dy$ and
$H(z)=\int_{z_L}^z(z-y)s(y)dy$. Then
$$\phi(z)=H(z)-\frac{z-z_L}{z_R-z_L}H(z_R),\qquad
E(z)=-G(z)+\frac{H(z_R)}{z_R-z_L}.$$
The solver precomputes these linear integration maps; it does not invert a
Poisson differentiation matrix. Charge density and the electric field are
updated at every RK4 stage. Velocity moments also use BSPF integration.

The kinetic solver uses resolved tensor weak forms, spatial upwind reservoir
fluxes, and the full nonlinear electric force. It evolves $f-f_M$ internally
to preserve the homogeneous reservoir equilibrium exactly. The default time
step is 0.02; this explicit method still requires transport stability checks.


In [ ]:
density_perturbation = b.integrate(vp, delta.transpose(2, 1, 0)).T
phi_check, electric_check = b.poisson_dirichlet(zp, density_perturbation.T)
field_map_error = float(jnp.max(jnp.abs(electric-electric_check.T)))
gauss_residual = float(jnp.max(jnp.abs(
    b.differentiate(zp, electric.T)+density_perturbation.T)))
print(f"Direct primitive vs precomputed field: {field_map_error:.3e}")
print(f"Differentiated Gauss-law residual:    {gauss_residual:.3e}")
assert field_map_error < 1e-12 and gauss_residual < 1e-7


## Distinguish phase mixing from boundary escape

Let $h=f-f_M$. In these normalized units the perturbation free energy is
$$\mathcal F=W_E+\iint[f\log(f/f_M)-f+f_M],dz,dv,
\qquad W_E=\tfrac12\int E^2dz.$$
For small perturbations the particle term is $\iint h^2/(2f_M)$. It can grow
as electric-field energy moves into velocity-space structure without collisions.
It is not a claim of irreversible entropy production.

We compute boundary free-energy flux through **all four phase-space faces**.
A second check integrates the actual electron work $-\int E\int vf\,dv\,dz$.
For grounded endpoints, $\int E dz=0$; there is no applied voltage work.
Finite velocity truncation can introduce additional small charge-loss effects,
so both balances are measured rather than assumed exact.


In [ ]:
def moment(field):
    return b.integrate(zp, b.integrate(vp, field.transpose(2, 1, 0)))

def primitive(signal):
    time_plan = b.plan_1d(times, degree=7, n_basis=24, boundary_points=9)
    return b.antiderivative(time_plan, signal)

ratio = delta/model.background
entropy_factor = (1+ratio)*jnp.log1p(ratio)-ratio
entropy_factor = jnp.where(jnp.abs(ratio) < 1e-4,
    ratio**2/2-ratio**3/6+ratio**4/12-ratio**5/20, entropy_factor)
free_density = model.background*entropy_factor
particle_free_energy = moment(free_density)
free_energy = field_energy+particle_free_energy
boundary_power = (b.integrate(vp, ((free_density[:, 0]-free_density[:, -1])*v).T)
    +b.integrate(zp, (-electric*(free_density[:, :, 0]-free_density[:, :, -1])).T))
boundary_transfer = primitive(boundary_power)
particle_flow = b.integrate(vp, (delta*v).transpose(2, 1, 0)).T
field_to_particles = primitive(-b.integrate(zp, (electric*particle_flow).T))
free_balance = float(jnp.max(jnp.abs(free_energy-free_energy[0]-boundary_transfer))/free_energy[0])
exchange_balance = float(jnp.max(jnp.abs(field_energy-field_energy[0]+field_to_particles))/field_energy[0])
escaped_fraction = float(-boundary_transfer[-1]/free_energy[0])
print(f"Initial free energy:             {float(free_energy[0]):.6e}")
print(f"Free energy lost at boundaries:  {escaped_fraction:.3%}")
print(f"Relative free-energy balance:    {free_balance:.3e}")
print(f"Field–particle exchange balance: {exchange_balance:.3e}")
assert free_balance < 1e-4 and exchange_balance < 1e-5
assert 0 <= escaped_fraction < 0.05
assert float(field_energy[-1]/field_energy[0]) < 0.25


## Resolution and finite-domain sensitivity

Refine spatial samples from 65 to 129, velocity samples from 129 to 257, and
halve dt. Then move grounded reservoirs from ±40 to ±60 while keeping the
same perturbation, velocity domain and spatial spacing. Compare the fields
on the shared interval. Electrostatic boundaries act through an elliptic field
solve, so there is no strict “boundary-free until particles arrive” guarantee.

All differences below are normalized by the maximum baseline electric field.
This checks the finite-domain simulation; it does not validate a fitted decay
rate against a single periodic Fourier mode.


In [ ]:
zc, _, _, _, _, _, _, ec = solve(nz=65)
_, _, _, _, _, _, _, ev = solve(nv=257)
_, _, _, _, _, _, _, et = solve(substeps=10)
zb, _, _, _, _, _, _, eb = solve(nz=193, half_length=60.)
scale = jnp.max(jnp.abs(electric))
space_change = float(jnp.max(jnp.abs(ec-electric[:, ::2]))/scale)
velocity_change = float(jnp.max(jnp.abs(ev-electric)))/float(scale)
time_change = float(jnp.max(jnp.abs(et-electric))/scale)
inside = (zb >= z[0]) & (zb <= z[-1])
domain_change = float(jnp.max(jnp.abs(eb[:, inside]-electric))/scale)
print(f"65 -> 129 spatial field change:   {space_change:.3e}")
print(f"129 -> 257 velocity field change: {velocity_change:.3e}")
print(f"Halved-step field change:         {time_change:.3e}")
print(f"Larger-domain field change:       {domain_change:.3e}")
assert space_change < 1e-4 and velocity_change < 1e-5 and time_change < 1e-5
assert domain_change < 0.01


## Interpretation

The diagnostic target is loss of electric-field energy accompanied by growth
of particle free energy and velocity-space phase mixing, with boundary loss
accounted for. The envelope is not a single exponential: the localized packet
contains a range of wavenumbers with different damping rates, including more
weakly damped long wavelengths. Boundary effects are small but measurable in
this time window. The model remains nonlinear, although the chosen amplitude
is small. Longer-time claims require additional velocity resolution and checks
for numerical recurrence; they cannot be inferred from this run.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8), constrained_layout=True)
axes[0, 0].semilogy(times, field_energy/field_energy[0])
axes[0, 0].set(xlabel="t", ylabel="Electric energy / initial", title="Damped oscillations of a localized packet")
limit = float(jnp.max(jnp.abs(delta[-1])))
image = axes[0, 1].pcolormesh(z, v, delta[-1].T, shading="auto", cmap="RdBu_r", vmin=-limit, vmax=limit)
axes[0, 1].set(xlabel="z", ylabel="Parallel velocity", ylim=(-4, 4), title="Phase mixing: f − Maxwellian at t = 12")
fig.colorbar(image, ax=axes[0, 1], label="Distribution perturbation")
axes[1, 0].plot(times, field_energy/free_energy[0], label="Electric field")
axes[1, 0].plot(times, particle_free_energy/free_energy[0], label="Particle free energy")
axes[1, 0].plot(times, free_energy/free_energy[0], label="Total")
axes[1, 0].plot(times, 1+boundary_transfer/free_energy[0], "--", label="Initial + boundary transfer")
axes[1, 0].set(xlabel="t", ylabel="Fraction of initial free energy", title="Field energy transfers into particles")
axes[1, 0].legend()
for index in (0, 30, 60, 120):
    axes[1, 1].plot(z, electric[index], label=f"t={float(times[index]):g}")
axes[1, 1].set(xlabel="z", ylabel="Self-consistent E", title="Finite interval with grounded endpoints")
axes[1, 1].legend()
plt.show()
